# UNI embedding

adapted from: https://github.com/uhlerlab/spatialfusion/blob/main/workflows/unimodal-embeddings/scripts/unimodal-embeddings.py

In [ ]:
import argparse
import logging
import os
import pathlib as pl
import sys
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import tifffile
import timm
import torch
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

: 

In [ ]:
def load_UNI_model(model_path: str, device: str = "cuda"):
    timm_kwargs = {
        'model_name': 'vit_giant_patch14_224',
        'img_size': 224,
        'patch_size': 14,
        'depth': 24,
        'num_heads': 24,
        'init_values': 1e-5,
        'embed_dim': 1536,
        'mlp_ratio': 2.66667 * 2,
        'num_classes': 0,
        'no_embed_class': True,
        'mlp_layer': timm.layers.SwiGLUPacked,
        'act_layer': torch.nn.SiLU,
        'reg_tokens': 8,
        'dynamic_img_size': True
    }

    model = timm.create_model(pretrained=False, **timm_kwargs)
    model.load_state_dict(torch.load(model_path, map_location="cpu"), strict=True)
    model.eval().to(device)

    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406),
                             std=(0.229, 0.224, 0.225)),
    ])
    return model, transform

In [ ]:
def embed_UNI(
    wsi, # whole-slide image as a NumPy array
    adata, # AnnData object; obs_names align with he_coords
    he_coords, # typically adata.obsm['spatial'], shape (n_spots, 2) with (x,y) pixel coords
    
    output_dir: str, # where to write embeddings, UNI.parquet
    model_path: str, # path to UNI weights
    output_name: str = "UNI.parquet",
    batch_size: int = 128, # number of patches per forward pass
    device:str = "cuda", # "cuda" or "cpu"
):

    print('Load UNI model')
    model, transform = load_UNI_model(model_path, device)

    os.makedirs(output_dir, exist_ok=True)

    embeddings = []
    cell_ids = []
    
    batch_imgs = []
    batch_ids = []
    
    print(f"Embedding {len(he_coords)} image patches in batches of {batch_size}...")
    for cid, (x, y) in tqdm(zip(adata.obs_names, he_coords), total=len(adata)):
        x, y = int(x), int(y)
        x0, x1 = x - 128, x + 128
        y0, y1 = y - 128, y + 128
    
        pad_x0 = max(0, -x0)
        pad_x1 = max(0, x1 - wsi.shape[1])
        pad_y0 = max(0, -y0)
        pad_y1 = max(0, y1 - wsi.shape[0])
    
        patch = np.pad(
            wsi[max(0, y0):min(wsi.shape[0], y1), max(0, x0):min(wsi.shape[1], x1)],
            ((pad_y0, pad_y1), (pad_x0, pad_x1), (0, 0)),
            mode="constant"
        )

        count = 0
        if patch.shape[:2] != (256, 256):
            if count < 5:  # Limit the number of warnings
                print(f"Skipped patch at ({x},{y}): shape={patch.shape}")
            count += 1
            continue
    
        tensor_img = transform(Image.fromarray(patch))
        batch_imgs.append(tensor_img)
        batch_ids.append(cid)
    
        if len(batch_imgs) == batch_size:
            img_tensor = torch.stack(batch_imgs).to(device)
    
            autocast_device = "cuda" if device == "cuda" else "cpu"
            with torch.inference_mode(), torch.autocast(device_type=autocast_device, dtype=torch.float16):
                batch_embs = model(img_tensor).to(torch.float16).cpu().numpy()
    
            embeddings.extend(batch_embs)
            cell_ids.extend(batch_ids)
            batch_imgs.clear()
            batch_ids.clear()
    
    # Final batch (if any)
    if batch_imgs:
        img_tensor = torch.stack(batch_imgs).to(device)

        autocast_device = "cuda" if device == "cuda" else "cpu"
        with torch.inference_mode(), torch.autocast(device_type=autocast_device, dtype=torch.float16):
            batch_embs = model(img_tensor).to(torch.float16).cpu().numpy()

        embeddings.extend(batch_embs)
        cell_ids.extend(batch_ids)

    # Save embedding matrix
    df = pd.DataFrame(embeddings, index=cell_ids)
    uni_out = pl.Path(output_dir) / output_name
    df.to_parquet(uni_out)
    print(f"Saved {len(df)} embeddings to {uni_out}") 

In [ ]:
import torch
print(torch.version.cuda)

In [ ]:
# loading args

wsi_path = "/insomnia001/depts/morpheus/users/me2982/data/xenium_ovca_full/ovca_he_image.ome.tif"
adata_path = "/insomnia001/depts/morpheus/users/me2982/data/xenium_ovca_full/ovca_test.h5ad"

# convering wsi and adata
adata = sc.read_h5ad(adata_path)
with tifffile.TiffFile(wsi_path) as tif:
            wsi = tif.series[0].asarray()

he_coords = adata.obsm["spatial"]
output_dir = "/insomnia001/depts/morpheus/users/me2982/data/xenium_ovca_full/embeddings"
model_path = "/insomnia001/depts/morpheus/users/me2982/models/pytorch_model.bin"
ouput_name = "UNI_test.parquet"
batch_size = 128
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

embed_UNI(
            wsi,
            adata,
            he_coords,
            output_dir,
            model_path,
            output_name=ouput_name,
            batch_size=batch_size,
            device=device,
        )